# Imports and load data

Imports libraries, defines shared plot settings, and loads the pre-fitted TRF results produced by `runTRFs.ipynb`.

- **Color scheme** — uses seaborn's `"deep"` palette; attended = blue, ignored = orange, null = gray. A custom blue–white–red diverging colormap is built for topographic maps.
- **`VARIABLES`** — a configuration dict passed to every plotting function. Key entries:
  - `bold_ch` — frontocentral scalp channels highlighted in bold on butterfly plots.
  - `bold_ave` — whether to overlay the average of `bold_ch` as a thick line.
  - `alpha` — significance threshold (0.05) for the masked-difference test.
  - `regressors` — the two acoustic predictors: temporal envelope (`~gammatone-1`) and onset envelope (`~gammatone-on-1`).
- **Data loading** — reads `results/sustA.pkl`, `results/switA.pkl`, and `results/convA.pkl`. Missing files are skipped with a warning.

In [ ]:
from pathlib import Path
import eelbrain as eel
import seaborn as sns
from plotting import plot_trfs, plot_topo, plot_masked_difference, plot_corr, get_yminmax, plot_trfs_convA, plot_optLag
from matplotlib.colors import LinearSegmentedColormap

ROOT = Path.cwd()
TRF_DIR = ROOT / 'results'
FIG_DIR = ROOT / 'figures'
FIG_DIR.mkdir(exist_ok=True)

# === Colors and colormaps ===
cc = sns.color_palette("deep")
colors = [cc[0], cc[1], cc[7]]
red, blue = cc[3], cc[0]
deep_diverging = LinearSegmentedColormap.from_list("blue_red", [blue, (1, 1, 1), red], N=256)


VARIABLES = {
    'condition': "",
    'trf_dir': "",
    'sensor_type': "",
    'fig_dir': "",
    'bold_ch': ['Fp1','Fp2','AFz','F3','Fz','F4','FC3','FC1','FC2','FC4','Cz','C3','C1','C2','C4','CP1','CPz','CP2'],
    'bold_ave': [True],
    'save_figs': True,
    'alpha': 0.05,
    'regressors': ['~gammatone-1', '~gammatone-on-1'],
    'colors': colors,
    'cmap': deep_diverging,
    'conv_channels': ['POz','FC3','TP10'],
}


conditions = ['sustA','switA','convA']  

df = {}
for condition in conditions:
    df[condition] = {}
    fname = TRF_DIR / f'{condition}.pkl'
    if fname.exists():
        df[condition] = eel.load.unpickle(fname)
    else:
        print(f'File {fname} not found. Skipping loading for condition {condition}.')


# Make forward TRF plots

Generates forward-model TRF visualisations for all three conditions (sustA, switA, convA) and both sensor arrays. For each condition and sensor type the following figures are produced (scalp only):

- **Butterfly plot** (`plot_trfs`) — all-channel TRF traces for attended, ignored, and null (control) predictors, with highlighted frontocentral channels overlaid in bold.
- **Topomap panel** (`plot_topo`) — scalp topographies at automatically detected peak latencies for the attended and ignored TRFs.
- **Masked difference** (`plot_masked_difference`) — attended-minus-ignored TRF difference masked by a TFCE-corrected independent-samples t-test at `alpha = 0.05`.
- **Correlation boxplot** (`plot_corr`) — grouped boxplot of backward-model *r* values (Target / Masker / Control) across scalp and cEEGrid sensor arrays.

Figures are saved as both `.svg` and `.png` under `figures/{condition}/`.

In [ ]:
# === Main Loop ===
for condition in conditions:
    VARIABLES['condition'] = condition

    fig_dir = FIG_DIR / condition
    fig_dir.mkdir(exist_ok=True)
    VARIABLES['fig_dir'] = fig_dir

    if condition == 'convA':
        trfs = df[condition]['trfs']['allTrials']
        corr = df[condition]['correlations']['allTrials']
    else:
        trfs = df[condition]['trfs']
        corr = df[condition]['correlations']

    for trf_dir in trfs.keys():
        VARIABLES['trf_dir'] = trf_dir
        for sensor_type in trfs[trf_dir].keys():
            VARIABLES['sensor_type'] = sensor_type
  
            if trf_dir == 'fw':
                ndvars = {}
                for attention in trfs[trf_dir][sensor_type]:
                    for regressor in trfs[trf_dir][sensor_type][attention]:
                        if regressor not in ndvars:
                            ndvars[regressor] = []
                        ndvars[regressor].append(trfs[trf_dir][sensor_type][attention][regressor][0])

                for regressor in list(ndvars.keys())[:2]:
                    if sensor_type == 'scalp':
                        print('skip for now :)')

                        plot_trfs(ndvars, regressor, VARIABLES)

                        plot_topo(ndvars, regressor, VARIABLES)

                        plot_masked_difference(ndvars, regressor, VARIABLES)

         
    plot_corr(corr['bw'], VARIABLES)



# Make backward correlation plots

Plots grouped boxplots of the backward-model cross-validated correlations for each condition. Each figure shows four sensor × predictor groups (Scalp-Envelope, Scalp-Onset, cEEGrid-Envelope, cEEGrid-Onset), with three boxes per group representing Target (attended), Masker (ignored), and Control (null/shifted) conditions.

The attended correlation should be significantly higher than the null, confirming that the EEG tracks the acoustic envelope of the attended speech stream. Figures are saved to `figures/{condition}/`.

In [ ]:
for condition in conditions:
    VARIABLES['condition'] = condition

    fig_dir = FIG_DIR / condition
    fig_dir.mkdir(exist_ok=True)
    VARIABLES['fig_dir'] = fig_dir

    if condition == 'convA':
        corr = df[condition]['correlations']['allTrials']
    else:
        corr = df[condition]['correlations']

    plot_corr(corr['bw'], VARIABLES)

# Make conversational vs single speaker plot

Plots forward TRFs specifically for the conversational attention (convA) condition, comparing the two conversation configurations: **front conversation** (attended talker is the front speaker) and **side talker** (attended talker is at the side).

Unlike the sustA/switA plots, this cell uses `plot_trfs_convA` which produces a 2×2 panel figure (front/side × attended/ignored) with individual-channel traces shown faintly and three highlighted channels (`POz`, `FC3`, `TP10`) overlaid in bold. Time axis is in milliseconds.

The y-axis limits are derived automatically from the global amplitude range across all NDVars using `get_yminmax`, with small manual adjustments to remove edge artefacts.

In [ ]:
condition = 'convA'
VARIABLES['convSession'] = 'convSingle'
trf_dir = 'fw'
sensor_type = 'scalp'
VARIABLES['trf_dir'] = trf_dir
VARIABLES['sensor_type'] = sensor_type

ndvars = {}
minmax = {}
for convSingle in df[condition]['trfs']['convSingle'][trf_dir][sensor_type].keys():
    for reg in VARIABLES['regressors']:
        if reg not in ndvars:
            ndvars[reg] = {}
            minmax[reg] = []
        if convSingle not in ndvars[reg]:
            ndvars[reg][convSingle] = {}
        for attention in df[condition]['trfs']['convSingle'][trf_dir][sensor_type][convSingle][reg].keys():
            if attention not in ndvars[reg][convSingle]:
                ndvars[reg][convSingle][attention] = []
            ndvar = df[condition]['trfs']['convSingle'][trf_dir][sensor_type][convSingle][reg][attention][0]

            ndvars[reg][convSingle][attention].append(ndvar)
            ndvar = ndvar.mean(axis='case')
            minmax[reg].append(ndvar)


for regressor in ndvars.keys():
    
    data_ndvar = ndvars[regressor]
    ymin, ymax = get_yminmax(minmax[regressor])

    VARIABLES['ymin'] = ymin + 0.0005
    VARIABLES['ymax'] = ymax - 0.0015

    plot_trfs_convA(data_ndvar, regressor, VARIABLES)

# Make optimal lag plot

Plots the optimal-lag analysis curves computed by `runTRFs.ipynb`. For each condition and acoustic regressor, `plot_optLag` shows mean correlation as a function of the TRF window's centre lag for both forward and backward models, with 95 % confidence intervals shaded.

- **Backward model** (left panel) — peak lag indicates the stimulus-to-EEG delay at which the decoder performs best; expected around 100–200 ms for auditory cortical responses.
- **Forward model** (right panel) — peak lag reflects the TRF latency that best predicts the acoustic predictor from the EEG; complementary to the backward result.

Separate curves are shown for attended and ignored conditions, for both scalp and cEEGrid sensor arrays. Figures are saved to `figures/{condition}/`.

In [ ]:
for condition in conditions:
    fig_dir = FIG_DIR / condition
    VARIABLES['fig_dir'] = fig_dir
    VARIABLES['condition'] = condition

    if condition == 'convA':
        optLag = df[condition]['corr_optLag']['allTrials']
    else:
        optLag = df[condition]['corr_optLag']

    for regs in VARIABLES['regressors']:
        plot_optLag(optLag, regs, VARIABLES)